# 🎙️ Textless Tamil Speech-to-Speech AI — Google Colab Training
*Inspired by Block's Dessa, Kyutai's Moshi, and Meta's AudioCraft*

This notebook trains a **100% textless Speech-to-Speech conversational model for the Tamil language** directly on continuous streaming audio from Hugging Face (`SPRINGLab/IndicVoices-R_Tamil`).

### Why Streaming Audio?
- **Zero Disk Bottleneck:** No need to download 50GB+ of audio datasets.
- **24 kHz On-The-Fly Resampling:** Streams audio directly into the Neural Audio Codec (Encodec RVQ).
- **Accelerated with WSP:** Uses pre-grafted Weight-Space Programming foundation weights (94.26M parameters) so the model converges rapidly on Colab GPU.

### Step 1: Check GPU Acceleration

In [ ]:
!nvidia-smi

### Step 2: Clone Repository & Install Dependencies

In [ ]:
!git clone https://github.com/sathiyaseelank-dot/tamil-voice-ai.git
%cd tamil-voice-ai
!pip install -q datasets soundfile librosa encodec transformers accelerate einops

### Step 2b: (Recommended) Mount Google Drive to Save Checkpoints Permanently
Google Colab's default storage is temporary (ephemeral). Mounting Google Drive guarantees that trained checkpoints (`.pt`) and sample generated Tamil voice files (`.wav`) are preserved permanently even if Colab disconnects.

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/tamil_voice_ai_checkpoints'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f"✓ Google Drive mounted! Checkpoints will persist in: {SAVE_DIR}")
except Exception as e:
    SAVE_DIR = 'checkpoints_streaming'
    print(f"Running without Drive mount. Saving locally to: {SAVE_DIR}")

### Step 3: Stream and Preview Real Tamil Audio from Hugging Face

In [ ]:
import IPython.display as ipd
from datasets import load_dataset, Audio

# Streaming the cleaned Tamil speech dataset
dataset = load_dataset(
    "SPRINGLab/IndicVoices-R_Tamil", 
    split="train", 
    streaming=True
)

# Resample audio on-the-fly to 24,000 Hz to match model's sample rate
dataset = dataset.cast_column("audio", Audio(sampling_rate=24000))

# Preview the first sample
sample = next(iter(dataset))
print("Transcript:", sample.get("text", sample.get("sentence", "")))
print("Audio array shape:", sample["audio"]["array"].shape)

# Listen to sample audio
ipd.Audio(sample["audio"]["array"], rate=24000)

### Step 4: Perform Weight-Space Programming (WSP) Surgery
Grafts 94.26M parameters from AudioCraft foundation weights in seconds.

In [ ]:
!python wsp_surgery.py

### Step 5: Start Streaming Training on GPU
Trains directly from the Hugging Face Tamil audio stream and automatically saves checkpoints to your mounted Google Drive.

In [ ]:
!python train_streaming.py \
    --dataset SPRINGLab/IndicVoices-R_Tamil \
    --checkpoint checkpoints/tamil_speech_lm_wsp_grafted.pt \
    --output_dir "$SAVE_DIR" \
    --max_steps 1000 \
    --lr 2e-4 \
    --device cuda \
    --log_every 10 \
    --save_every 200

### Step 6: Listen to Generated Tamil Speech Response

In [ ]:
import glob
import IPython.display as ipd

generated_wavs = sorted(glob.glob(f"{SAVE_DIR}/*.wav"))
if generated_wavs:
    latest_wav = generated_wavs[-1]
    print(f"Playing latest generated Tamil response: {latest_wav}")
    ipd.display(ipd.Audio(latest_wav))
else:
    # Fallback to WSP verification audio
    ipd.display(ipd.Audio("wsp_tamil_response.wav"))

### Step 7: Interactive Tamil Voice Inference (Anti-Humming & Crisp Speech)
Condition on a REAL spoken Tamil sentence and use Nucleus (Top-p) Sampling with Repetition Penalty to eliminate monotone humming.

In [ ]:
from inference import TamilVoiceAIInference
import soundfile as sf
import IPython.display as ipd

# 1. Extract a REAL spoken Tamil sentence as prompt
real_sample = next(iter(dataset))
print("Prompt Transcript:", real_sample.get('text', real_sample.get('sentence', '')))
sf.write("real_tamil_prompt.wav", real_sample["audio"]["array"], 24000)

# 2. Run inference with anti-humming sampling
engine = TamilVoiceAIInference(
    checkpoint_path=f"{SAVE_DIR}/tamil_speech_lm_stream_step_1000.pt",
    device="cuda"
)
engine.respond_to_speech(
    "real_tamil_prompt.wav",
    output_audio_path="crisp_tamil_response.wav",
    max_duration_sec=2.5,
    temperature=0.35,
    top_k=40,
    top_p=0.85,
    repetition_penalty=1.25
)

# 3. Listen to Prompt and AI Reply
print("\n--- Input Prompt Audio ---")
ipd.display(ipd.Audio("real_tamil_prompt.wav"))
print("--- AI Generated Spoken Tamil Response ---")
ipd.display(ipd.Audio("crisp_tamil_response.wav"))